# Nemotron 3 Nano 4B: safety-drift pilot (Colab)

Runs the full study from the `nemotron-safety-drift` repo on one Colab GPU:
scan → judge → LoRA fine-tune on benign data → re-measure → release gate → evidence packages → Substack draft.

**Before you start**
1. *Runtime → Change runtime type →* a GPU (T4 works; L4 or A100 is much faster if offered).
2. In the 🔑 **Secrets** panel (left sidebar), add `HF_TOKEN` and `NVIDIA_API_KEY` with *Notebook access* on.
3. Run the cells in order, or *Runtime → Run all*.

Everything is saved to Google Drive (`MyDrive/nsd_runs`). If Colab disconnects, reconnect and run all again: finished stages are skipped and partial generations resume.

## 1. GPU and Google Drive

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → choose a GPU"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
print("bf16 supported:", torch.cuda.is_bf16_supported())
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code
Either set `REPO_URL` to your GitHub repo, or put `nemotron-safety-drift.zip` in the top of your Google Drive (or leave both empty and you'll be asked to upload the zip).

In [ ]:
import os, zipfile, shutil, pathlib
REPO_URL = ""   # e.g. "https://github.com/<you>/nemotron-safety-drift"
DRIVE_ZIP = "/content/drive/MyDrive/nemotron-safety-drift.zip"
REPO = pathlib.Path("/content/nemotron-safety-drift")
if not REPO.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    else:
        src = DRIVE_ZIP
        if not os.path.exists(src):
            from google.colab import files
            src = list(files.upload().keys())[0]
        zipfile.ZipFile(src).extractall("/content")
os.chdir(REPO)
print("code at", REPO)

## 3. Install dependencies (about 5–15 minutes)

In [ ]:
%pip install -q peft accelerate datasets fastapi uvicorn openai jinja2 pyyaml matplotlib
# garak gets its own environment so its pins can't disturb the model stack.
%pip install -q uv
!uv venv -q /content/garak-venv && uv pip install -q --python /content/garak-venv/bin/python garak==0.17.0
# Optional fused Mamba kernels: much faster generation and training for the hybrid Mamba-2 layers.
# Building can take 10+ minutes; if it fails, everything still runs on the slower PyTorch path.
INSTALL_MAMBA_KERNELS = True
if INSTALL_MAMBA_KERNELS:
    r = subprocess.run("pip install -q --no-build-isolation mamba-ssm causal-conv1d", shell=True)
    print("mamba kernels:", "installed" if r.returncode == 0 else "not installed (slow path)")

## 4. Keys from Colab Secrets

In [ ]:
from google.colab import userdata
for k in ["HF_TOKEN", "NVIDIA_API_KEY"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception as e:
        raise SystemExit(f"Add {k} in the Secrets panel (key icon) and enable Notebook access: {e}")
print("keys loaded (values not shown)")
os.environ.update(NSD_CONFIG="config/lite_4b.yaml", NSD_WORKDIR="/content/drive/MyDrive/nsd_runs",
                  TRAIN_PY="python", GARAK_PYTHON="/content/garak-venv/bin/python", PYTHONPATH=str(REPO / "src"))

## 5. Smoke test (about 5 minutes)
Loads the 4B model, answers one prompt, checks the hosted guard, and estimates throughput. If this fails, stop here and share the error.

In [ ]:
import time, json, urllib.request
!bash scripts/serve.sh target nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
def ask(prompt, max_tokens=128):
    body = json.dumps({"model": "target", "messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens,
                       "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}).encode()
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", body, {"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=600).read())
t = time.time(); r = ask("In two sentences, what is master data management?")
dt = time.time() - t
print(r["choices"][0]["message"]["content"])
print(f"\n{r['usage']['completion_tokens']} tokens in {dt:.1f}s for one request (batched runs are much faster per prompt)")
!bash scripts/serve.sh stop
!python -c "import asyncio; from nsd import config, judge; asyncio.run(judge.calibrate(config.load()))"

## 6. Run the study
Long-running (hours on a T4; less on L4/A100). Keep this tab open; Colab may disconnect idle sessions. If it does, reconnect and *Run all*: it resumes.

In [ ]:
!bash run.sh 2>&1 | grep -v "it/s]"

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
RUN = pathlib.Path(os.environ["NSD_WORKDIR"]) / "nemotron3-nano-4b-lora-drift-lite-v1"
display(Markdown((RUN / "results/report.md").read_text()))
for png in sorted((RUN / "results").glob("*.png")):
    display(Image(str(png)))
print("Substack draft:", RUN / "results/substack_post.md")
print("Evidence packages:", sorted(str(p) for p in (RUN / "evidence").glob("*/evidence.json")))